# IBM Data Science Professional Certificate Capstone Project

## EDA using Python’s SQLite3 module

### Overview

We will use SQLite3 to execute SQL queries seamlessly while leveraging Python for advanced data analysis. Below are the steps to set up and run SQL queries in a Jupyter Notebook.

### Download the datasets

In [ ]:
# Install sqlalchemy version 1.3.9 to ensure compatibility with ipython-sql
%pip install sqlalchemy==1.3.9

### Connect to the database

Let's load the SQL extension and establish a connection with the database

In [ ]:
#Install the ipython-sql package to enable SQL support in Jupyter notebooks
#!pip install ipython-sql

In [ ]:
# Load the SQL extension in Jupyter notebook
%load_ext sql

In [ ]:
# Connect to the SQLite database
import csv, sqlite3

con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [ ]:
# Install pandas 1.1.5
%pip install -q pandas==1.1.5

In [ ]:
# Connect to my_data1.db using ipython-sql
%sql sqlite:///my_data1.db

'Connected: @my_data1.db'

In [ ]:
# Read the data from the csv file and load the data into a new table in the SQLite database
import pandas as pd
df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")

In [ ]:
# Create table SPACEXTABLE with only the records where Date is not null
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///my_data1.db
Done.


[]

### Step 1
Display the names of the unique launch sites in the space mission.

In [10]:
%sql SELECT DISTINCT "Launch_Site" FROM SPACEXTABLE;

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40



### Step 2
Display 5 records where launch sites begin with the string 'CCA'.

In [11]:
%sql SELECT "Launch_Site" FROM SPACEXTABLE WHERE "Launch_Site" LIKE "CCA%" LIMIT 5;

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
CCAFS LC-40
CCAFS LC-40
CCAFS LC-40
CCAFS LC-40


### Step 3
Display the total payload mass carried by boosters launched by NASA (CRS).

In [ ]:
%sql SELECT customer, SUM(PAYLOAD_MASS__KG_) AS total_payload FROM SPACEXTABLE WHERE customer="NASA (CRS)" GROUP BY customer;

 * sqlite:///my_data1.db
Done.


Customer,total_payload
NASA (CRS),45596


### Step 4
Display average payload mass carried by booster version F9 v1.1.

In [15]:
%sql SELECT "Booster_Version", AVG(PAYLOAD_MASS__KG_) AS avg_payload  FROM SPACEXTABLE WHERE "Booster_Version"="F9 v1.1" GROUP BY "Booster_Version";

 * sqlite:///my_data1.db
Done.


Booster_Version,avg_payload
F9 v1.1,2928.4


### Step 5
List the date when the first succesful landing outcome in ground pad was acheived.


In [16]:
%sql SELECT "Landing_Outcome", MIN("Date") AS min_date FROM SPACEXTABLE WHERE "Landing_Outcome"="Success (ground pad)" GROUP BY "Landing_Outcome";

 * sqlite:///my_data1.db
Done.


Landing_Outcome,min_date
Success (ground pad),2015-12-22


### Step 6
List the names of the boosters which have success in drone ship and have payload mass greater than 4000 but less than 6000.

In [19]:
%sql SELECT DISTINCT "Booster_Version" FROM SPACEXTABLE WHERE "Landing_Outcome"="Success (drone ship)" AND (PAYLOAD_MASS__KG_>4000 AND PAYLOAD_MASS__KG_<6000) ;

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


### Step 7
List the total number of successful and failure mission outcomes.

In [21]:
%sql SELECT "Mission_Outcome", COUNT(*) AS count_misssion_outcome FROM SPACEXTABLE GROUP BY TRIM("Mission_Outcome");

 * sqlite:///my_data1.db
Done.


Mission_Outcome,count_misssion_outcome
Failure (in flight),1
Success,99
Success (payload status unclear),1


### Step 8
List the   names of the booster_versions which have carried the maximum payload mass.

In [ ]:
%sql SELECT "Booster_Version" FROM SPACEXTABLE WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) AS max_payload FROM SPACEXTABLE);

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 B5 B1048.4
F9 B5 B1049.4
F9 B5 B1051.3
F9 B5 B1056.4
F9 B5 B1048.5
F9 B5 B1051.4
F9 B5 B1049.5
F9 B5 B1060.2
F9 B5 B1058.3
F9 B5 B1051.6


### Step 9
List the records which will display the month names, failure landing_outcomes in drone ship, booster versions, launch_site for the months in year 2015.

Note: SQLLite does not support monthnames. Using substr(Date, 6,2) as month to get the months and substr(Date,0,5)='2015' for year.

In [34]:
%sql  SELECT "Date" FROM SPACEXTABLE LIMIT 5;

 * sqlite:///my_data1.db
Done.


Date
2010-06-04
2010-12-08
2012-05-22
2012-10-08
2013-03-01


In [36]:
%sql  SELECT substr("Date",1,4) AS year, substr("Date",6,2) AS month, "Landing_Outcome", "Booster_Version", "Launch_Site" FROM SPACEXTABLE WHERE "Landing_Outcome"= "Failure (drone ship)" AND substr("Date",0,5)='2015';

 * sqlite:///my_data1.db
Done.


year,month,Landing_Outcome,Booster_Version,Launch_Site
2015,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
2015,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Step 10
Rank the count of landing outcomes (such as Failure (drone ship) or Success (ground pad)) between the date 2010-06-04 and 2017-03-20, in descending order.

In [38]:
%sql  SELECT Count("Landing_Outcome") AS count_landing_outcomes FROM SPACEXTABLE WHERE ("Landing_Outcome" = "Failure (drone ship)" OR "Landing_Outcome" = "Success (ground pad)") AND ("Date" BETWEEN "2010-06-04" AND "2017-03-20");

 * sqlite:///my_data1.db
Done.


count_landing_outcomes
8
